# 🌾 Rice Disease Detection — Colab + ngrok Launcher
Run the full FastAPI + Gradio app from Google Colab with a public ngrok URL.

**Steps:**
1. Upload your trained model files to `/content/trained_models/`
2. Run cells top-to-bottom
3. Click the ngrok URL at the bottom to open the demo


In [ ]:
# ── Cell 1 — Mount Drive + copy project files ────────────────────────────────
# Option A: project is already on your Drive
from google.colab import drive
drive.mount("/content/drive")

import shutil, os

# Adjust this path to where you cloned/uploaded the project on Drive
PROJECT_ON_DRIVE = "/content/drive/MyDrive/rice-disease-detection"

if os.path.exists(PROJECT_ON_DRIVE):
    if os.path.exists("/content/rice-disease-detection"):
        shutil.rmtree("/content/rice-disease-detection")
    shutil.copytree(PROJECT_ON_DRIVE, "/content/rice-disease-detection")
    print("✓ Project copied from Drive")
else:
    print("Project not found on Drive — upload or clone it manually.")


In [ ]:
# ── Cell 2 — Copy trained models into project folder ─────────────────────────
# Your trained model files should be on Drive from Colab training sessions.
# Adjust the source paths below to match YOUR Drive folder names.

import shutil, os

MODELS_ON_DRIVE = {
    "/content/drive/MyDrive/Updated_rice_disease/rice_v3_efficientnet.keras" : "rice_cnn_model.keras",
    "/content/drive/MyDrive/best_lstm_v2.pth"                                : "rice_lstm_model.pth",
    "/content/drive/MyDrive/scaler.pkl"                                      : "scaler.pkl",
}

dest_dir = "/content/rice-disease-detection/trained_models"
os.makedirs(dest_dir, exist_ok=True)

for src, dest_name in MODELS_ON_DRIVE.items():
    dest = os.path.join(dest_dir, dest_name)
    if os.path.exists(src):
        shutil.copy(src, dest)
        size_mb = os.path.getsize(dest) / 1e6
        print(f"  ✓ {dest_name}  ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ NOT FOUND: {src}  ← update path above")


In [ ]:
# ── Cell 3 — Install dependencies ────────────────────────────────────────────
# Colab already has TF, PyTorch, numpy, pandas, sklearn, matplotlib.
# We only need to install the packages that are NOT pre-installed.

import subprocess, sys

EXTRA_PACKAGES = [
    "fastapi>=0.104.0",
    "uvicorn[standard]>=0.24.0",
    "python-multipart>=0.0.6",
    "pyngrok>=7.0.0",
    "gradio>=4.0.0",
    "openmeteo-requests>=0.2.0",
    "requests-cache>=1.1.0",
    "retry-requests>=2.0.0",
    "PyYAML>=6.0",
]

print("Installing extra packages...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet"] + EXTRA_PACKAGES,
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    print("✓ All packages installed")


In [ ]:
# ── Cell 4 — Set ngrok auth token ────────────────────────────────────────────
# Get your free token from https://dashboard.ngrok.com/get-started/your-authtoken
# Free account allows 1 tunnel at a time — enough for demo.

from pyngrok import ngrok, conf

NGROK_AUTH_TOKEN = "PASTE_YOUR_TOKEN_HERE"   # ← replace this

if NGROK_AUTH_TOKEN == "PASTE_YOUR_TOKEN_HERE":
    raise ValueError("Set your ngrok auth token above before running this cell.")

conf.get_default().auth_token = NGROK_AUTH_TOKEN
print("✓ ngrok token set")


In [ ]:
# ── Cell 5 — Start the app ────────────────────────────────────────────────────
import subprocess, threading, time, sys, os

os.chdir("/content/rice-disease-detection")
sys.path.insert(0, "/content/rice-disease-detection")

PORT = 7860

# Kill any previous uvicorn on that port
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
time.sleep(1)

# Start uvicorn in background thread
def run_server():
    subprocess.run([
        sys.executable, "-m", "uvicorn",
        "app.main:app",
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--workers", "1",
    ])

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for startup (model loading takes ~15–30 seconds)
print("Waiting for app to start (loading models)...")
import urllib.request, time
for i in range(60):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2)
        print(f"✓ App is up after {i+1}s")
        break
    except:
        time.sleep(1)
        if i % 10 == 9:
            print(f"  still loading... ({i+1}s)")
else:
    print("⚠ App did not respond in 60s — check logs above for errors")


In [ ]:
# ── Cell 6 — Open ngrok tunnel → get public URL ──────────────────────────────
from pyngrok import ngrok

# Close any existing tunnels first
ngrok.kill()

tunnel = ngrok.connect(7860, "http")
public_url = tunnel.public_url

print("=" * 60)
print(f"  🌐 PUBLIC URL  :  {public_url}")
print(f"  📋 Gradio UI   :  {public_url}/")
print(f"  📖 API Docs    :  {public_url}/docs")
print(f"  ❤️  Health      :  {public_url}/health")
print("=" * 60)
print("\nShare the PUBLIC URL with your college/external reviewers.")
print("The tunnel stays active as long as this Colab session is running.")


In [ ]:
# ── Cell 7 — Quick API smoke test (run after Cell 6) ─────────────────────────
# Tests the /health endpoint and CNN prediction endpoint locally.
# Requires a test image — adjust the path below.

import requests, json

BASE = "http://localhost:7860"   # test locally (faster than ngrok)

# Health check
resp = requests.get(f"{BASE}/health")
print("Health check:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

# CNN prediction test (update path to any leaf image you have)
TEST_IMAGE = "/content/data/raw/test/Brown Spot/IMG_20231018_143715.jpg"
if os.path.exists(TEST_IMAGE):
    with open(TEST_IMAGE, "rb") as f:
        resp = requests.post(
            f"{BASE}/api/v1/predict/image",
            files={"file": ("leaf.jpg", f, "image/jpeg")},
        )
    print("\nCNN prediction:", resp.status_code)
    if resp.ok:
        r = resp.json()
        print(f"  Predicted: {r['predicted_class']}  ({r['confidence']*100:.1f}%)")
        print(f"  CNN risk : {r['cnn_risk_score']:.4f}")
    else:
        print(resp.text)
else:
    print(f"\n⚠ Test image not found at {TEST_IMAGE} — update path")


In [ ]:
# ── Cell 8 — Stop tunnel (run when done) ─────────────────────────────────────
from pyngrok import ngrok
ngrok.kill()
print("✓ ngrok tunnel closed")
